# EarXplore Authentication – Build Timeline Matrices

This notebook builds:
- **coauthor_matrix_auth.csv** (1 if two studies share at least one author name)
- **citation_matrix_auth.csv** (placeholder all-zeros; replace if you have citation data)

It reads your authentication CSV and writes the matrices into a separate folder so it won't overwrite the interaction matrices.

In [1]:
import os
import pandas as pd
import numpy as np


In [2]:
# ====== CONFIG ======
# Put your authentication CSV filename here (relative to this notebook), or give an absolute path.
AUTH_CSV = "authentication_fixed.csv" 

# Output folder (kept separate from interaction matrices)
OUT_DIR = "interconnections_datasets_auth"
os.makedirs(OUT_DIR, exist_ok=True)

COAUTHOR_OUT = os.path.join(OUT_DIR, "coauthor_matrix_auth.csv")
CITATION_OUT = os.path.join(OUT_DIR, "citation_matrix_auth.csv")


In [3]:
# ====== LOAD DATA ======
df = pd.read_csv(AUTH_CSV)
df = df.fillna("N/A").replace("nan", "N/A")

# Require these columns
required = {"ID", "Authors"}
missing = sorted(list(required - set(df.columns)))
if missing:
    raise ValueError(f"Missing required columns in {AUTH_CSV}: {missing}")

df_id_authors = df[["ID", "Authors"]].copy()
df_id_authors.head()


,ID,Authors
0,48,"Akkermans, A.H.M. and Kevenaar, T.A.M. and Sch..."
1,36,"Liu, Yuxi and Hatzinakos, Dimitrios"
2,47,"Arakawa, Takayuki and Koshinaka, Takafumi and ..."
3,44,"Curran, Max T. and Yang, Jong-kai and Merrill,..."
4,43,"Derawi, Mohammad"


In [4]:
# ====== BUILD CO-AUTHOR MATRIX ======
# Connect two studies if they share at least one EXACT normalized author name.
# Assumes 'Authors' is a comma-separated string (like "Last, First and ..., ...")
# If your CSV uses a different delimiter, adjust split logic below.

ids = df_id_authors["ID"].astype(int).to_numpy()

def normalize_name(name: str) -> str:
    # lowercase + collapse internal whitespace
    return " ".join(str(name).strip().lower().split())

def to_author_set(value) -> set:
    if isinstance(value, str):
        # split on commas that separate authors (same as your interaction notebook)
        names = [n for n in (x.strip() for x in value.split(",")) if n]
    else:
        names = []
    return {normalize_name(n) for n in names}

id_to_authors = {int(row["ID"]): to_author_set(row["Authors"]) for _, row in df_id_authors.iterrows()}

# Create a square matrix with study IDs as both index + columns
coauthor_matrix = pd.DataFrame(0, index=ids, columns=ids, dtype=int)

for i, id_i in enumerate(ids):
    a_i = id_to_authors.get(int(id_i), set())
    for j in range(i + 1, len(ids)):
        id_j = ids[j]
        a_j = id_to_authors.get(int(id_j), set())
        if a_i and a_j and (a_i & a_j):
            coauthor_matrix.loc[id_i, id_j] = 1
            coauthor_matrix.loc[id_j, id_i] = 1  # symmetric

coauthor_matrix.to_csv(COAUTHOR_OUT)
COAUTHOR_OUT, coauthor_matrix.shape, coauthor_matrix.head()


('interconnections_datasets_auth\\coauthor_matrix_auth.csv',
 (46, 46),
     48  36  47  44  43  45  35  34  38  46  ...  5   1   32  29  26  23  19  \
 48   0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   
 36   0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   
 47   0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   
 44   0   0   0   0   0   1   0   0   0   0  ...   0   0   0   0   0   0   0   
 43   0   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   
 
     15  7   3   
 48   0   0   0  
 36   1   0   0  
 47   0   0   0  
 44   0   0   0  
 43   0   0   0  
 
 [5 rows x 46 columns])